<a href="https://colab.research.google.com/github/Magistrate-dot/ML-AI-project/blob/main/Minecraft_Mob_Detection_model_Comparison_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prep

In [ ]:
!pip install -q ultralytics huggingface_hub scikit-learn
import os
import json
import shutil
from collections import defaultdict
from sklearn.model_selection import train_test_split

from huggingface_hub import snapshot_download
DATASET_PATH = "/content/minecraft_dataset"

repo_dir = snapshot_download(
    repo_id="twoturtles/minecraft-mobs",
    repo_type="dataset"
)

print(repo_dir)

for split in ["train", "valid"]:
    os.makedirs(f"{DATASET_PATH}/{split}/images", exist_ok=True)
    os.makedirs(f"{DATASET_PATH}/{split}/labels", exist_ok=True)

with open(os.path.join(repo_dir, "annotations.json"), "r") as f:
    coco = json.load(f)

images = coco["images"]
annotations = coco["annotations"]

annotations_by_image = defaultdict(list)

for ann in annotations:
    annotations_by_image[ann["image_id"]].append(ann)

train_images, valid_images = train_test_split(
    images,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

print("Training images:", len(train_images))
print("Validation images:", len(valid_images))

def process_split(image_list, split_name):

    for img in image_list:

        image_id = img["id"]
        filename = img["file_name"]

        src = os.path.join(repo_dir, "images", filename)
        dst = os.path.join(DATASET_PATH, split_name, "images", filename)

        shutil.copy(src, dst)

        label_file = os.path.join(
            DATASET_PATH,
            split_name,
            "labels",
            filename.replace(".png", ".txt")
        )

        with open(label_file, "w") as f:

            for ann in annotations_by_image.get(image_id, []):

                x, y, w, h = ann["bbox"]

                xc = (x + w / 2) / img["width"]
                yc = (y + h / 2) / img["height"]
                wn = w / img["width"]
                hn = h / img["height"]

                cls = ann["category_id"]

                f.write(
                    f"{cls} {xc} {yc} {wn} {hn}\n"
                )
process_split(train_images, "train")
process_split(valid_images, "valid")

yaml_text = f"""
path: {DATASET_PATH}

train: train/images
val: valid/images

names:
  0: chicken
  1: cow
  2: creeper
  3: enderman
  4: pig
  5: sheep
  6: skeleton
  7: spider
  8: zombie
"""

with open("data.yaml", "w") as f:
    f.write(yaml_text)

print("Train images:", len(os.listdir(f"{DATASET_PATH}/train/images")))
print("Train labels:", len(os.listdir(f"{DATASET_PATH}/train/labels")))

print("Validation images:", len(os.listdir(f"{DATASET_PATH}/valid/images")))
print("Validation labels:", len(os.listdir(f"{DATASET_PATH}/valid/labels")))



# Model

In [ ]:
from ultralytics import YOLO

nano = YOLO("yolo11n.pt")
nano.train(
    data="data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
)

small = YOLO("yolo11s.pt")
small.train(
    data="data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
)

# Analytics

In [ ]:
from ultralytics import YOLO
import pandas as pd
import matplotlib.pyplot as plt

print(" Extracting metrics from the 50-epoch run folders...")

# Point to the exact best weights generated by your latest run
nano_metrics = YOLO("/content/runs/detect/train/weights/best.pt").val(split="val")
small_metrics = YOLO("/content/runs/detect/train-2/weights/best.pt").val(split="val")

# Generate the data frame
comparison = pd.DataFrame({
    "Model": ["YOLO11 Nano", "YOLO11 Small"],
    "Precision": [nano_metrics.box.mp, small_metrics.box.mp],
    "Recall": [nano_metrics.box.mr, small_metrics.box.mr],
    "mAP50": [nano_metrics.box.map50, small_metrics.box.map50],
    "mAP50-95": [nano_metrics.box.map, small_metrics.box.map]
})

print("\n=== MODEL PERFORMANCE COMPARISON ===")
print(comparison)

# Save out to CSV for spreadsheet software
comparison.to_csv("final_50_epoch_comparison.csv", index=False)

# Plot the mAP50 performance side-by-side
plt.figure(figsize=(6, 4))
plt.bar(comparison["Model"], comparison["mAP50"], color=['#4CAF50', '#2196F3'])
plt.title("Minecraft Mob Detection: 50-Epoch Model Comparison (mAP50)")
plt.ylabel("mAP50 Score")
plt.ylim(0, 1.0)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

# Test

In [ ]:
import os
from ultralytics import YOLO
from IPython.display import Image, display

# 1. Path to your uploaded JPG image inside the content folder
image_path = "/content/Test_mob.jpg"  # Changed extension to .jpg

# 2. Path to your 50-epoch Small model weights
model_path = "/content/runs/detect/train-2/weights/best.pt"

# Quick check to save you from errors
if not os.path.exists(image_path):
    print(f" Error: Could not find your image at {image_path}. Please make sure your file in the 'content' folder is named exactly 'test_mob.jpg'!")
elif not os.path.exists(model_path):
    print(f" Error: Could not find your model weights at {model_path}. Double-check your sidebar to see if the folder name is 'train' or 'train-2'.")
else:
    print(" Loading your trained YOLO11 Small brain...")
    model = YOLO(model_path)

    print(f" Analyzing '{os.path.basename(image_path)}' for Minecraft mobs...")
    # Run prediction and save visually
    results = model.predict(source=image_path, conf=0.25, save=True)

    # Grab the output paths dynamically
    saved_dir = results[0].save_dir
    base_name = os.path.basename(results[0].path)
    output_visual_path = os.path.join(saved_dir, base_name)

    print("\n🎉 DETECTION COMPLETE! Here is what your AI found:")
    display(Image(filename=output_visual_path))